### **1. Runtime & Environment Initialization**

In [ ]:
# IMPORTANT:
# In Google Colab:
# Runtime → Change runtime type → Select "T4 GPU"
# This ensures hardware acceleration for model training.
# --------------------------------------------------------------

# --------------------------------------------------------------
# Install required dependencies
# transformers  → Hugging Face library for pretrained LLMs
# torch         → PyTorch deep learning framework
# gradio        → Web UI for interactive chatbot demos
# datasets      → Hugging Face dataset management library
# --------------------------------------------------------------


In [ ]:
!pip install transformers torch gradio datasets

In [ ]:
import torch

# AutoModelForCausalLM → Loads GPT-style models for text generation
# AutoTokenizer        → Converts text ↔ tokens (model-readable format)
# Trainer / TrainingArguments → Used for fine-tuning workflows
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments
)

# load_dataset → Download and manage NLP datasets easily
from datasets import load_dataset

# Gradio → Used to build a lightweight web interface
import gradio as gr


In [ ]:
# --------------------------------------------------------------
# Device Configuration (CPU / GPU)
# --------------------------------------------------------------
# If CUDA (NVIDIA GPU support) is available → use GPU
# Otherwise fallback to CPU

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

## **2. Baseline Chatbot (DialoGPT Inference)**

In [ ]:
# Load pretrained DialoGPT model + tokenizer
# - DialoGPT = Dialogue Generative Pre-trained Transformer
# - Causal LM (Causal Language Model): predicts next token left→right
# - Tokenizer: converts text ↔ token IDs (numbers the model understands)
# --------------------------------------------------------------

MODEL_NAME = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# GPT-style models often have no PAD token by default.
# We reuse EOS (End Of Sequence) token as PAD to avoid warnings/errors.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id

# Move model to GPU if available (device was created earlier)
model = model.to(device)
model.eval()  # evaluation mode (disables dropout)

In [ ]:
# --------------------------------------------------------------
# Baseline chatbot function (multi-turn)
# - We keep "chat_history_ids" to preserve short context
# - Each new user message is appended to the previous conversation
# --------------------------------------------------------------

chat_history_ids = None

def chatbot_response(user_input: str, chat_history_ids=None):
    """
    Generate a response using DialoGPT with optional conversation history.

    Parameters:
    - user_input: user message (string)
    - chat_history_ids: previous generated tokens to keep context (optional)

    Returns:
    - response: bot text reply
    - chat_history_ids: updated conversation token history
    """
    user_input = (user_input or "").strip()
    if not user_input:
        return "Please enter a message", chat_history_ids

    # Encode the new user input and add EOS token (end of message marker)
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"
    ).to(device)

    # Append to chat history if it exists (keeps conversation context)
    bot_input_ids = (
        torch.cat([chat_history_ids, new_input_ids], dim=-1)
        if chat_history_ids is not None
        else new_input_ids
    )

    # Generate model output
    with torch.no_grad():
        chat_history_ids = model.generate(
            bot_input_ids,
            max_length=1000,  # total length including history
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the newly generated part (exclude the prompt tokens)
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    ).strip()

    # Safety fallback for empty responses
    if not response:
        response = "I’m not sure how to answer that."

    return response, chat_history_ids


## **Launch Your First Chatbot Locally**

In [ ]:
css = """
/* ===== Global page ===== */
.gradio-container {
  font-family: ui-sans-serif, system-ui, -apple-system, Segoe UI, Roboto, Arial;
  background: radial-gradient(1200px circle at 10% 10%, #ffe3ef 0%, #fff7f3 35%, #eef5ff 100%);
}

/* ===== Main card ===== */
.gradio-container .wrap {
  max-width: 920px !important;
  margin: 40px auto !important;
}

.gradio-container .panel,
.gradio-container .block,
.gradio-container .form {
  border-radius: 18px !important;
}

/* ===== Title ===== */
h1, .gradio-container h1 {
  text-align: center;
  font-size: 40px !important;
  line-height: 1.1;
  font-weight: 800 !important;
  margin-bottom: 18px !important;
  color: #1f2a37 !important;
  letter-spacing: -0.02em;
}

h1::after{
  content: "";
  display: block;
  width: 110px;
  height: 6px;
  margin: 12px auto 0;
  border-radius: 999px;
  background: linear-gradient(90deg, #ff4d8d, #ff9f5a);
  opacity: 0.9;
}

/* ===== Blocks (input/output containers) ===== */
.gradio-container .block {
  background: rgba(255,255,255,0.65) !important;
  border: 1px solid rgba(31,41,55,0.08) !important;
  box-shadow: 0 10px 30px rgba(31,41,55,0.08) !important;
  backdrop-filter: blur(10px);
  padding: 18px !important;
}

/* Labels */
label, .gradio-container label {
  font-weight: 700 !important;
  color: #374151 !important;
  font-size: 14px !important;
  letter-spacing: 0.02em;
}

/* ===== Inputs / Textareas ===== */
input[type="text"], textarea {
  background: rgba(255,255,255,0.9) !important;
  border: 1px solid rgba(31,41,55,0.14) !important;
  border-radius: 14px !important;
  padding: 14px 14px !important;
  font-size: 16px !important;
  color: #111827 !important;
  outline: none !important;
  box-shadow: none !important;
}

input[type="text"]::placeholder, textarea::placeholder {
  color: rgba(17,24,39,0.45) !important;
}

/* Focus glow */
input[type="text"]:focus, textarea:focus {
  border-color: rgba(255,77,141,0.65) !important;
  box-shadow: 0 0 0 4px rgba(255,77,141,0.14) !important;
}

/* ===== Output area ===== */
.gradio-container .output_text,
.gradio-container .wrap .prose,
.gradio-container .wrap textarea[readonly] {
  background: linear-gradient(180deg, rgba(255,255,255,0.95), rgba(255,255,255,0.75)) !important;
  border: 1px solid rgba(31,41,55,0.12) !important;
  border-radius: 14px !important;
  color: #111827 !important;
}

/* ===== Buttons ===== */
button, .gradio-container button {
  border-radius: 14px !important;
  padding: 12px 16px !important;
  font-weight: 800 !important;
  font-size: 16px !important;
  border: none !important;
  transition: transform 0.15s ease, filter 0.2s ease, box-shadow 0.2s ease !important;
}

/* Primary button (Submit) */
.gradio-container button.primary,
.gradio-container .primary button,
.gradio-container .gr-button-primary {
  background: linear-gradient(90deg, #ff4d8d, #ff9f5a) !important;
  color: white !important;
  box-shadow: 0 10px 18px rgba(255,77,141,0.18) !important;
}

/* Secondary button (Clear) */
.gradio-container .secondary button,
.gradio-container .gr-button-secondary {
  background: rgba(255,255,255,0.85) !important;
  color: #111827 !important;
  border: 1px solid rgba(31,41,55,0.14) !important;
}

/* Hover */
button:hover {
  transform: translateY(-1px) !important;
  filter: brightness(1.02);
  box-shadow: 0 14px 26px rgba(31,41,55,0.12) !important;
}

/* ===== Footer ===== */
footer {
  text-align: center;
  margin-top: 18px;
  font-size: 13px;
  color: rgba(17,24,39,0.55);
}
"""

In [ ]:
# ==============================================================
# 3. Interactive Demo: Gradio Web UI
# ==============================================================

# This UI keeps conversation memory using gr.State()
# so the chatbot can respond in multi-turn mode.

# Optional: built-in theme (clean + modern)
theme = gr.themes.Soft(
    primary_hue="pink",
    secondary_hue="orange",
    neutral_hue="slate",
)

def gradio_chat(user_input, state_chat_history):
    """
    Gradio wrapper around chatbot_response().

    Inputs:
    - user_input: text from the user
    - state_chat_history: stored token history (chat_history_ids)

    Outputs:
    - response: bot reply
    - updated_state: updated token history
    """
    response, updated_state = chatbot_response(user_input, state_chat_history)
    return response, updated_state

with gr.Blocks(theme=theme, css=css) as demo:
    gr.Markdown("# Baseline Chatbot (DialoGPT)")
    gr.Markdown(
        "A minimal multi-turn chatbot demo using **DialoGPT** + **Gradio**. "
        "Conversation context is stored in session state."
    )

    # State for chat history tokens (stored across interactions)
    chat_state = gr.State(None)

    user_input = gr.Textbox(
        placeholder="Type your message…",
        lines=2,
        label="User input"
    )

    output = gr.Textbox(label="Bot response")

    with gr.Row():
        submit = gr.Button("Send", variant="primary")
        clear = gr.Button("Clear", variant="secondary")

    # When user clicks Send
    submit.click(
        fn=gradio_chat,
        inputs=[user_input, chat_state],
        outputs=[output, chat_state],
    )

    # Enter key should also send
    user_input.submit(
        fn=gradio_chat,
        inputs=[user_input, chat_state],
        outputs=[output, chat_state],
    )

    # Clear everything
    def reset():
        return "", "", None

    clear.click(
        fn=reset,
        inputs=[],
        outputs=[user_input, output, chat_state]
    )

demo.launch()

## **4. Fine-Tuning DialoGPT (Training Pipeline))**

In [ ]:
# We use a dialogue dataset to adapt the model to better conversation patterns.
# Dataset used here: blended_skill_talk (multi-turn conversational data).

from datasets import load_dataset

DATASET_NAME = "blended_skill_talk"
dataset = load_dataset(DATASET_NAME)

# Optional: quick, clean preview (1 example only)
print("Dataset:", dataset)
print("Sample keys (train):", dataset["train"].column_names)

# For faster training in Colab, we fine-tune on a subset (10% here).
train_data = dataset["train"].shuffle(seed=42).select(range(len(dataset["train"]) // 10))
valid_data = dataset["validation"].shuffle(seed=42).select(range(len(dataset["validation"]) // 10))

print(f"Train subset size: {len(train_data)} | Validation subset size: {len(valid_data)}")

In [ ]:
# --------------------------------------------------------------
# Tokenization
# - We transform conversations into a single training text sequence
# - DialoGPT is a Causal LM: it learns to predict next tokens
# --------------------------------------------------------------

from transformers import DataCollatorForLanguageModeling

# Ensure PAD token exists for GPT-style models
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

MAX_LENGTH = 256

def tokenize_function(examples):
    """
    blended_skill_talk uses 'free_messages' as a list of utterances per conversation.
    We join the utterances into one text string to train the model on multi-turn flow.
    """
    text_list = [
        " </s> ".join(msgs) if isinstance(msgs, list) else str(msgs)
        for msgs in examples["free_messages"]
    ]
    return tokenizer(text_list, truncation=True, max_length=MAX_LENGTH)

tokenized_train = train_data.map(
    tokenize_function,
    batched=True,
    remove_columns=train_data.column_names
)

tokenized_valid = valid_data.map(
    tokenize_function,
    batched=True,
    remove_columns=valid_data.column_names
)

# Data collator handles dynamic padding during batching
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # mlm=False because this is causal language modeling (not masked LM)
)

In [ ]:
import transformers
print("transformers version:", transformers.__version__)

In [ ]:
# --------------------------------------------------------------
# Training configuration
# --------------------------------------------------------------

from transformers import Trainer, TrainingArguments

OUTPUT_DIR = "./fine_tuned_chatbot"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,

    # evaluation_strategy is the correct argument name
    eval_strategy="steps",
    eval_steps=500,

    logging_steps=50,
    save_steps=500,
    save_total_limit=2,

    # Use mixed precision if GPU is available (faster on T4)
    fp16=False,

    # Avoid writing to external loggers by default
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
)

trainer.train()

# Save final artifacts (model + tokenizer)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Fine-tuned model saved to: {OUTPUT_DIR}")

## **5. Inference on the Fine-Tuned Model + Simple UI (Gradio)**
this is a single turn execute this block or the next one, choose juste one

In [ ]:
css = """
/* ===== Global page ===== */
.gradio-container {
  font-family: ui-sans-serif, system-ui, -apple-system, Segoe UI, Roboto, Arial;
  background: radial-gradient(1200px circle at 10% 10%, #ffe3ef 0%, #fff7f3 35%, #eef5ff 100%);
}

/* ===== Main card ===== */
.gradio-container .wrap {
  max-width: 920px !important;
  margin: 40px auto !important;
}

.gradio-container .panel,
.gradio-container .block,
.gradio-container .form {
  border-radius: 18px !important;
}

/* ===== Title ===== */
h1, .gradio-container h1 {
  text-align: center;
  font-size: 40px !important;
  line-height: 1.1;
  font-weight: 800 !important;
  margin-bottom: 18px !important;
  color: #1f2a37 !important;
  letter-spacing: -0.02em;
}

h1::after{
  content: "";
  display: block;
  width: 110px;
  height: 6px;
  margin: 12px auto 0;
  border-radius: 999px;
  background: linear-gradient(90deg, #ff4d8d, #ff9f5a);
  opacity: 0.9;
}

/* ===== Blocks (input/output containers) ===== */
.gradio-container .block {
  background: rgba(255,255,255,0.65) !important;
  border: 1px solid rgba(31,41,55,0.08) !important;
  box-shadow: 0 10px 30px rgba(31,41,55,0.08) !important;
  backdrop-filter: blur(10px);
  padding: 18px !important;
}

/* Labels */
label, .gradio-container label {
  font-weight: 700 !important;
  color: #374151 !important;
  font-size: 14px !important;
  letter-spacing: 0.02em;
}

/* ===== Inputs / Textareas ===== */
input[type="text"], textarea {
  background: rgba(255,255,255,0.9) !important;
  border: 1px solid rgba(31,41,55,0.14) !important;
  border-radius: 14px !important;
  padding: 14px 14px !important;
  font-size: 16px !important;
  color: #111827 !important;
  outline: none !important;
  box-shadow: none !important;
}

input[type="text"]::placeholder, textarea::placeholder {
  color: rgba(17,24,39,0.45) !important;
}

/* Focus glow */
input[type="text"]:focus, textarea:focus {
  border-color: rgba(255,77,141,0.65) !important;
  box-shadow: 0 0 0 4px rgba(255,77,141,0.14) !important;
}

/* ===== Output area ===== */
.gradio-container .output_text,
.gradio-container .wrap .prose,
.gradio-container .wrap textarea[readonly] {
  background: linear-gradient(180deg, rgba(255,255,255,0.95), rgba(255,255,255,0.75)) !important;
  border: 1px solid rgba(31,41,55,0.12) !important;
  border-radius: 14px !important;
  color: #111827 !important;
}

/* ===== Buttons ===== */
button, .gradio-container button {
  border-radius: 14px !important;
  padding: 12px 16px !important;
  font-weight: 800 !important;
  font-size: 16px !important;
  border: none !important;
  transition: transform 0.15s ease, filter 0.2s ease, box-shadow 0.2s ease !important;
}

/* Primary button (Submit) */
.gradio-container button.primary,
.gradio-container .primary button,
.gradio-container .gr-button-primary {
  background: linear-gradient(90deg, #ff4d8d, #ff9f5a) !important;
  color: white !important;
  box-shadow: 0 10px 18px rgba(255,77,141,0.18) !important;
}

/* Secondary button (Clear) */
.gradio-container .secondary button,
.gradio-container .gr-button-secondary {
  background: rgba(255,255,255,0.85) !important;
  color: #111827 !important;
  border: 1px solid rgba(31,41,55,0.14) !important;
}

/* Hover */
button:hover {
  transform: translateY(-1px) !important;
  filter: brightness(1.02);
  box-shadow: 0 14px 26px rgba(31,41,55,0.12) !important;
}

/* ===== Footer ===== */
footer {
  text-align: center;
  margin-top: 18px;
  font-size: 13px;
  color: rgba(17,24,39,0.55);
}
"""
print("✅ CSS loaded.")

In [ ]:
# Safety (GPT-style models)
FINETUNED_DIR = "./fine_tuned_chatbot"

tokenizer = AutoTokenizer.from_pretrained(FINETUNED_DIR)
model = AutoModelForCausalLM.from_pretrained(FINETUNED_DIR)

# GPT-style models sometimes don't have PAD token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# CRITICAL: avoid CUDA vocab mismatch
model.resize_token_embeddings(len(tokenizer))

model.config.pad_token_id = tokenizer.pad_token_id
model = model.to(device)
model.eval()

print("Model loaded successfully.")

theme = gr.themes.Soft(
    primary_hue="pink",
    secondary_hue="orange",
    neutral_hue="slate",
)

# --------------------------------------------------------------
# Chat Function (Multi-turn + Safe)
# --------------------------------------------------------------

@torch.no_grad()
def chatbot_response(user_input, chat_history_ids):

    user_input = (user_input or "").strip()
    if not user_input:
        return "Please enter a message ", chat_history_ids

    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"
    ).to(device)

    bot_input_ids = (
        torch.cat([chat_history_ids, new_input_ids], dim=-1)
        if chat_history_ids is not None
        else new_input_ids
    )

    # Prevent extremely long context (CUDA safety)
    MAX_CONTEXT_TOKENS = 800
    if bot_input_ids.shape[-1] > MAX_CONTEXT_TOKENS:
        bot_input_ids = bot_input_ids[:, -MAX_CONTEXT_TOKENS:]

    chat_history_ids = model.generate(
        bot_input_ids,
        max_new_tokens=60,
        do_sample=True,
        top_k=50,
        top_p=0.92,
        temperature=0.75,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    response = tokenizer.decode(
        chat_history_ids[0, bot_input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    if not response:
        response = "I’m not sure how to answer that."

    return response, chat_history_ids


# --------------------------------------------------------------
# Gradio UI (Same style as first one)
# --------------------------------------------------------------

theme = gr.themes.Soft(
    primary_hue="pink",
    secondary_hue="orange",
    neutral_hue="slate",
)

def reset():
    return "", "", None

with gr.Blocks(theme=theme, css=css) as demo:

    gr.Markdown("# Trained Chatbot (DialoGPT Fine-Tuned)")

    # Create state FIRST
    chat_state = gr.State(None)

    # UI Components
    user_input = gr.Textbox(
        placeholder="Type your message…",
        lines=2,
        label="User input"
    )

    bot_output = gr.Textbox(label="Bot response")

    with gr.Row():
        submit = gr.Button("Submit", variant="primary")
        clear = gr.Button("Clear", variant="secondary")

    # Connect function to buttons
    submit.click(
        fn=chatbot_response,
        inputs=[user_input, chat_state],
        outputs=[bot_output, chat_state],
    )

    user_input.submit(
        fn=chatbot_response,
        inputs=[user_input, chat_state],
        outputs=[bot_output, chat_state],
    )

    # Reset logic
    def reset():
        return "", "", None

    clear.click(
        fn=reset,
        inputs=[],
        outputs=[user_input, bot_output, chat_state],
    )

# Launch AFTER the block
demo.launch(debug=True)

### **5. Fine-Tuned Chatbot UI (Multi-turn with Memory)**

In [ ]:

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

model.eval()

@torch.no_grad()
def chatbot_response(user_input: str, chat_history_ids):
    user_input = (user_input or "").strip()
    if not user_input:
        return "Please enter a message", chat_history_ids

    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"
    ).to(model.device)

    bot_input_ids = (
        torch.cat([chat_history_ids, new_input_ids], dim=-1)
        if chat_history_ids is not None
        else new_input_ids
    )

    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        do_sample=True,
        top_k=50,
        top_p=0.92,
        temperature=0.75,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    response = tokenizer.decode(
        chat_history_ids[0, bot_input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    if not response:
        response = "I’m not sure how to answer that."

    return response, chat_history_ids


with gr.Blocks() as demo:
    gr.Markdown("# Trained Chatbot (DialoGPT Fine-Tuned)")
    gr.Markdown("Multi-turn chatbot with conversation memory stored in session state.")

    state = gr.State(None)
    inp = gr.Textbox(placeholder="Type your message…", lines=2, label="User input")
    out = gr.Textbox(label="Bot response")

    with gr.Row():
        send = gr.Button("Send")
        clear = gr.Button("Clear")

    send.click(chatbot_response, inputs=[inp, state], outputs=[out, state])
    inp.submit(chatbot_response, inputs=[inp, state], outputs=[out, state])

    def reset():
        return "", "", None

    clear.click(reset, inputs=[], outputs=[inp, out, state])

demo.launch()



### **6. RAG (Retrieval-Augmented Generation)**



In [ ]:
@torch.no_grad()
def chatbot_response(user_input, chat_history_ids):

    user_input = (user_input or "").strip()
    if not user_input:
        return "Please enter a message", chat_history_ids

    # -----------------------
    # 1.Retrieve context
    # -----------------------
    retrieved_info = retrieve_relevant_info(user_input)

    prefix = (retrieved_info + "\n") if retrieved_info else ""

    new_input_ids = tokenizer.encode(
        prefix + user_input + tokenizer.eos_token,
        return_tensors="pt"
    ).to(device)

    # -----------------------
    # 2. Append history
    # -----------------------
    bot_input_ids = (
        torch.cat([chat_history_ids, new_input_ids], dim=-1)
        if chat_history_ids is not None
        else new_input_ids
    )

    # Prevent extremely long context (CUDA safety)
    MAX_CONTEXT_TOKENS = 800
    if bot_input_ids.shape[-1] > MAX_CONTEXT_TOKENS:
        bot_input_ids = bot_input_ids[:, -MAX_CONTEXT_TOKENS:]

    # -----------------------
    # 3. Generate
    # -----------------------
    chat_history_ids = model.generate(
        bot_input_ids,
        max_new_tokens=60,
        do_sample=True,
        top_k=50,
        top_p=0.92,
        temperature=0.75,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    response = tokenizer.decode(
        chat_history_ids[0, bot_input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    if not response:
        response = "I’m not sure how to answer that."

    return response, chat_history_ids

### **7. Improving Response Coherence and Context Awareness**

In [ ]:
MAX_TURNS = 6               # number of user+bot turns to keep
MAX_CONTEXT_TOKENS = 800    # safety limit to avoid CUDA issues
conversation_history = []
@torch.no_grad()
def chatbot_response(user_input, state):
    """
    state is a dict that contains:
    - 'history_ids' (tensor) for DialoGPT history
    - 'turns' (list[str]) for human-readable conversation context
    """

    # Initialize state if first message
    if state is None:
        state = {"history_ids": None, "turns": []}

    user_input = (user_input or "").strip()
    if not user_input:
        return "Please enter a message", state

    # 1) Add user message to readable history
    state["turns"].append(f"User: {user_input}")

    # keep only last MAX_TURNS*2 lines (User+Bot)
    if len(state["turns"]) > MAX_TURNS * 2:
        state["turns"] = state["turns"][-MAX_TURNS * 2 :]

    # 2) Build prompt from turns (context)
    prompt = "\n".join(state["turns"]) + "\nBot:"

    # 3) Encode prompt
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    # Safety: avoid too-long prompt
    if input_ids.shape[-1] > MAX_CONTEXT_TOKENS:
        input_ids = input_ids[:, -MAX_CONTEXT_TOKENS:]

    # 4) Generate
    output_ids = model.generate(
        input_ids,
        max_new_tokens=60,
        do_sample=True,
        top_k=50,
        top_p=0.92,
        temperature=0.75,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    # 5) Decode only new tokens
    response = tokenizer.decode(
        output_ids[0, input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    if not response:
        response = "I’m not sure how to answer that."

    # 6) Add bot response to readable history
    state["turns"].append(f"Bot: {response}")

    return response, state


### **8.Fallback Mechanism (Handle Uncertain Responses)**

In [ ]:

@torch.no_grad()
def chatbot_response(user_input, state):

    # Initialize state if first message
    if state is None:
        state = []

    user_input = (user_input or "").strip()
    if not user_input:
        return "Please enter a message", state

    # Add user message to history
    state.append(f"User: {user_input}")

    # Keep only last 6 turns (light context memory)
    if len(state) > 6:
        state = state[-6:]

    prompt = "\n".join(state) + "\nBot:"

    input_ids = tokenizer.encode(
        prompt,
        return_tensors="pt"
    ).to(device)

    output_ids = model.generate(
        input_ids,
        max_new_tokens=50,
        do_sample=True,
        top_k=50,
        top_p=0.9,
        temperature=0.8,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id,
    )

    response = tokenizer.decode(
        output_ids[0, input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    # ------------------------------------------------
    # Fallback logic
    # ------------------------------------------------
    if (not response) or (len(response.split()) <= 2):
        response = "I'm not sure I understood that. Could you please rephrase?"

    # Add bot response to history
    state.append(f"Bot: {response}")

    return response, state